# Предсказание виральности треков

Можно ли по характеристикам трека предсказать, станет ли он вирусным, и что именно двигает трек в хиты?

**Данные:** `viral_songs.csv` — 10 000 треков, 17 признаков. Синтетические, учебные.
**Задача:** бинарная классификация (`viral` = 0/1).

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [4]:
df = pd.read_csv('viral_songs.csv')
df.head()

,track_id,track_name,artist,genre,release_year,duration_sec,danceability,energy,valence,tempo_bpm,acousticness,instrumentalness,speechiness,liveness,loudness_db,streams,viral
0,track_00001,Track_00001,Artist_1519,Pop,2013,237,0.760,0.703,0.747,140.8,0.115,0.103,0.206,0.272,-9.41,661412,1
1,track_00002,Track_00002,Artist_0458,Electronic,2018,239,0.436,0.934,0.672,105.1,0.016,0.002,0.024,0.000,-9.89,1114175,1
2,track_00003,Track_00003,Artist_1386,Hip-Hop,2014,148,0.642,0.725,0.460,90.6,0.010,0.520,0.079,0.053,-5.53,105832,1
3,track_00004,Track_00004,Artist_0865,Classical,2012,216,0.326,0.456,0.299,71.3,0.966,0.042,0.000,0.074,-11.39,915397,0
4,track_00005,Track_00005,Artist_0192,Pop,2011,277,0.599,0.683,0.700,132.6,0.078,0.113,0.000,0.036,-5.56,3690733,1


## 1. Осмотр данных

In [5]:
print(df.shape)
print(df['viral'].mean().round(3))
df.isna().sum()

(10000, 17)
0.36


,0
track_id,0
track_name,0
artist,0
genre,0
release_year,0
duration_sec,0
danceability,0
energy,0
valence,0
tempo_bpm,0


**Вывод:** 10 000 треков, пропусков нет. Виральных — 36%, то есть перекос в сторону невирусных (36/64). Значит accuracy сам по себе слаб (модель, предсказывающая всем «0», уже даёт 64%), и смотреть надо на **recall** и **ROC-AUC**.

## 2. EDA — виральность по жанрам

In [6]:
(df.groupby('genre')['viral'].mean()*100).round(1).sort_values()

,viral
genre,
Classical,1.2
Jazz,6.3
Indie,13.9
Country,23.9
Rock,32.7
R&B,40.5
Electronic,50.7
Hip-Hop,61.1
Pop,61.8


**Вывод:** разброс огромный — Classical 1.2%, а Latin/Pop/Hip-Hop ~61–65%. Жанр уже на глаз выглядит сильнейшим фактором. Это гипотеза, проверим моделью.

## 3. Признаки без утечки

In [7]:
# streams — СЛЕДСТВИЕ виральности (трек стал вирусным -> набрал прослушивания) = утечка, выкидываем.
# track_id / track_name / artist — идентификаторы, не признаки (artist = ~2000 уникальных).
drop = ['viral','streams','track_id','track_name','artist']
x = pd.get_dummies(df.drop(columns=drop), columns=['genre','release_year'], drop_first=True)
y = df['viral']

**Вывод:** `streams` исключён — это утечка (известен только ПОСЛЕ наступления виральности). Цель `viral` — 0/1, значит задача классификации, а не регрессии (линейная регрессия тут даёт отрицательный R²).

## 4. Логистическая регрессия

In [8]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train)   # fit только на train — без утечки
x_test_s = scaler.transform(x_test)

model = LogisticRegression(max_iter=1000)
model.fit(x_train_s, y_train)
y_pred = model.predict(x_test_s)
proba = model.predict_proba(x_test_s)[:, 1]

print(classification_report(y_test, y_pred))
print('ROC-AUC:', round(roc_auc_score(y_test, proba), 3))

              precision    recall  f1-score   support

           0       0.78      0.81      0.80      1279
           1       0.64      0.60      0.62       721

    accuracy                           0.74      2000
   macro avg       0.71      0.71      0.71      2000
weighted avg       0.73      0.74      0.73      2000

ROC-AUC: 0.807


**Вывод:** ROC-AUC = 0.807, recall класса 1 = 0.60. Accuracy 0.74 — выше базовой 64% («всё 0»), значит сигнал реальный. Виральность по свойствам трека предсказуема.

## 5. Драйверы — коэффициенты логрега

In [9]:
coef = pd.Series(model.coef_[0], index=x.columns).sort_values()
print('Снижают виральность:')
print(coef.head(5).round(2))
print('\nПовышают виральность:')
print(coef.tail(5).round(2))

Снижают виральность:
acousticness        -0.25
instrumentalness    -0.18
speechiness         -0.07
loudness_db         -0.04
release_year_2017   -0.03
dtype: float64

Повышают виральность:
genre_Electronic    0.54
genre_R&B           0.58
genre_Latin         0.66
genre_Pop           0.73
genre_Hip-Hop       0.80
dtype: float64


**Вывод:** сильнее всего **повышают** виральность жанры (Hip-Hop, Pop, Latin), **снижают** — `acousticness` и `instrumentalness`. Акустические и инструментальные треки реже залетают в хиты.

## 6. RandomForest — проверка + важность признаков

In [10]:
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(x_train, y_train)   # лесу масштабирование не нужно
rf_proba = rf.predict_proba(x_test)[:, 1]
print('RF ROC-AUC:', round(roc_auc_score(y_test, rf_proba), 3))

importance = pd.Series(rf.feature_importances_, index=x.columns).sort_values(ascending=False)
importance.head(10).round(3)

RF ROC-AUC: 0.792


,0
danceability,0.131
acousticness,0.126
energy,0.100
valence,0.081
tempo_bpm,0.069
loudness_db,0.067
duration_sec,0.065
liveness,0.063
instrumentalness,0.060
speechiness,0.059


**Вывод:** RF ROC-AUC = 0.796 — близко к логрегу (0.807), значит сигнал устойчив, не артефакт одной модели. Но важность у леса иная: вверху аудио-фичи (`danceability`, `acousticness`, `energy`), жанров в топе нет.

**Почему расходятся:** логрег оценивает вклад признака при прочих равных, и бинарный жанр даёт крупный отдельный сдвиг; лес считает важность по частоте использования признака в разбиениях, и непрерывные аудио-фичи дают больше точек разреза. Оба взгляда верны и дополняют друг друга.

## 7. Итог

**Виральность предсказуема** — две независимые модели дают близкий ROC-AUC (0.81 и 0.80).

**Жанр задаёт базовый шанс:** от 1% виральных у Classical до ~65% у Latin/Pop/Hip-Hop. **Аудио-свойства уточняют внутри жанра:** выше танцевальность/энергия — выше шанс, выше акустичность/инструментальность — ниже (согласованно в обеих моделях).

**Рекомендация для продвижения:** отбирать треки популярных жанров И с высокой танцевальностью/энергией, низкой акустичностью. Модель поймает ~60% будущих хитов (recall 0.60) — инструмент приоритизации, не замена экспертизе.

**Честные ограничения:** данные синтетические (выводы демонстрируют метод, не реальные закономерности музыки); логрег и лес по-разному ранжируют важность — показаны оба, а не выбран удобный.